In [ ]:
import sqlite3, pandas as pd, random
from datetime import date, timedelta

# -------------------------
# Step 1: Data Setup
# -------------------------
random.seed(42)
conn = sqlite3.connect(':memory:')

conn.execute('CREATE TABLE customers (id INT, name TEXT, region TEXT)')
conn.execute('CREATE TABLE orders (id INT, cust_id INT, amount REAL, order_date TEXT)')

for i in range(20):
    conn.execute('INSERT INTO customers VALUES (?,?,?)',
        (i, f'Customer_{i}', random.choice(['North','South','East','West'])))

for i in range(100):
    conn.execute('INSERT INTO orders VALUES (?,?,?,?)',
        (i, random.randint(0, 19), round(random.uniform(10, 300), 2),
         str(date(2024,1,1) + timedelta(days=random.randint(0, 364)))))

conn.commit()

# -------------------------
# Q1: Top 5 customers by total spend
# -------------------------
q1 = pd.read_sql_query("""
SELECT c.id, c.name, SUM(o.amount) AS total_spend
FROM customers c
JOIN orders o ON c.id = o.cust_id
GROUP BY c.id, c.name
ORDER BY total_spend DESC
LIMIT 5
""", conn)

# -------------------------
# Q2: Customers spending above average
# -------------------------
q2 = pd.read_sql_query("""
SELECT c.id, c.name, SUM(o.amount) AS total_spend
FROM customers c
JOIN orders o ON c.id = o.cust_id
GROUP BY c.id, c.name
HAVING total_spend > (
    SELECT AVG(total_spend)
    FROM (
        SELECT SUM(amount) AS total_spend
        FROM orders
        GROUP BY cust_id
    )
)
""", conn)

# -------------------------
# Q3: Orders from Q4 2024
# -------------------------
q3 = pd.read_sql_query("""
SELECT *
FROM orders
WHERE order_date >= '2024-10-01'
""", conn)

# -------------------------
# Validation
# -------------------------
assert len(q1) == 5, "Q1 should return 5 rows"
assert len(q2) >= 0, "Q2 should return results"
assert all(r['order_date'] >= '2024-10-01' for _, r in q3.iterrows()), "Q3 date filter failed"

print("All 5 checks passed!")

# Optional display
print("\nQ1:\n", q1)
print("\nQ2:\n", q2)
print("\nQ3 sample:\n", q3.head())